# PharmaLens AI — Project 19
## Commercial Finance & Budget Intelligence

FY2026 Budget Architecture, GTN & Net Price, Target Cascade, Regional Budgeting, Retail vs MOH Tender Economics, Price/Volume/Mix, Sales Force Budget, Marketing ROCI, Incentives, Reforecasting and FX Impact.

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',100)
rng=np.random.default_rng(42)
PROJECT_ROOT=os.path.abspath('..')
DATA_DIR=os.path.join(PROJECT_ROOT,'data')
os.makedirs(DATA_DIR,exist_ok=True)


In [ ]:
# Optional existing PharmaLens master data
candidates=[os.path.join(DATA_DIR,'cleaned_pharma_data.parquet'),os.path.join(PROJECT_ROOT,'data','processed','cleaned_pharma_data.parquet')]
master_data=None
for path in candidates:
    if os.path.exists(path):
        try:
            master_data=pd.read_parquet(path); print('Loaded:',path,master_data.shape); break
        except Exception as e: print('Load error:',e)
if master_data is None: print('No master parquet found; using synthetic finance data.')


## 1. Synthetic Commercial Finance Dataset

In [ ]:
brands=['Brand_A','Brand_B','Brand_C','Brand_D','Brand_E','Brand_F']
regions=['Cairo','Delta','Upper Egypt','Canal','Alexandria']
channels=['Retail','MOH Tender','Private Hospital','Distributor']
rows=[]
for brand in brands:
  for region in regions:
    for channel in channels:
      gross=rng.uniform(4e6,14e6); target=gross*rng.uniform(.90,1.18); price=rng.uniform(80,240)
      units=gross/price; gtn_rate=rng.uniform(.06,.18); cogs_rate=rng.uniform(.25,.55); spend=target*rng.uniform(.08,.25)
      rows.append([brand,region,channel,target,gross,price,units,gtn_rate,cogs_rate,spend])
finance=pd.DataFrame(rows,columns=['Brand','Region','Channel','Target_Gross_Sales','Actual_Gross_Sales','Avg_Price','Units','GTN_Rate','COGS_Rate','Commercial_Spend'])
finance['GTN']=finance.Actual_Gross_Sales*finance.GTN_Rate
finance['Net_Sales']=finance.Actual_Gross_Sales-finance.GTN
finance['COGS']=finance.Net_Sales*finance.COGS_Rate
finance['Contribution']=finance.Net_Sales-finance.COGS
finance['Target_Achievement']=finance.Actual_Gross_Sales/finance.Target_Gross_Sales
finance['ROCI']=(finance.Contribution-finance.Commercial_Spend)/finance.Commercial_Spend
finance.head()


## 2. Budget Architecture — Board Target = 480M EGP

In [ ]:
def build_budget(target_gross_sales,gtn_rate,cogs_rate,commercial_spend):
    gtn=target_gross_sales*gtn_rate; net=target_gross_sales-gtn; cogs=net*cogs_rate; contribution=net-cogs; after_spend=contribution-commercial_spend
    return pd.Series({'Target Gross Sales':target_gross_sales,'GTN':gtn,'Target Net Sales':net,'COGS':cogs,'Contribution':contribution,'Commercial Spend':commercial_spend,'Contribution After Spend':after_spend,'ROCI':after_spend/commercial_spend if commercial_spend else np.nan})
build_budget(480_000_000,.12,.38,105_600_000)


## 3. GTN & Net Price

In [ ]:
def gtn_net_price(gross_sales,units,gtn_rate):
    gtn=gross_sales*gtn_rate; net=gross_sales-gtn
    return pd.Series({'Gross Sales':gross_sales,'GTN':gtn,'Net Sales':net,'Gross Price':gross_sales/units if units else np.nan,'Net Price':net/units if units else np.nan,'GTN %':gtn_rate})
gtn_net_price(480_000_000,2_000_000,.12)


## 4. Target Cascade & Regional Budgeting

In [ ]:
regional=pd.DataFrame({'Region':regions,'Historical_Share':[.30,.22,.18,.12,.18],'Growth_Potential':[1.05,1.15,1.20,1.08,1.10],'Strategic_Weight':[1,1.05,1.10,.95,1]})
regional['Allocation_Score']=regional.Historical_Share*regional.Growth_Potential*regional.Strategic_Weight
regional['Budget_Share']=regional.Allocation_Score/regional.Allocation_Score.sum()
regional['Allocated_Target']=regional.Budget_Share*480_000_000
regional


## 5. Retail vs MOH Tender Economics

In [ ]:
channels_econ=pd.DataFrame({'Channel':['Retail','MOH Tender','Private Hospital','Distributor'],'List_Price_Index':[1,.78,.92,.86],'GTN_Rate':[.10,.04,.08,.12],'COGS_Rate':[.36,.34,.36,.38],'Commercial_Spend_Rate':[.16,.07,.12,.10]})
channels_econ['Net_Price_Index']=channels_econ.List_Price_Index*(1-channels_econ.GTN_Rate)
channels_econ['Contribution_Index']=channels_econ.Net_Price_Index*(1-channels_econ.COGS_Rate)
channels_econ['After_Spend_Index']=channels_econ.Contribution_Index-channels_econ.Commercial_Spend_Rate
channels_econ['ROCI_Index']=channels_econ.After_Spend_Index/channels_econ.Commercial_Spend_Rate
channels_econ


## 6. Price–Volume–Mix Analysis

In [ ]:
def price_volume_analysis(base_price,base_volume,new_price,new_volume):
    base=base_price*base_volume; new=new_price*new_volume
    return pd.Series({'Base Revenue':base,'New Revenue':new,'Price Effect':(new_price-base_price)*base_volume,'Volume Effect':(new_volume-base_volume)*base_price,'Price-Volume Interaction':(new_price-base_price)*(new_volume-base_volume),'Total Revenue Variance':new-base,'Revenue Variance %':(new-base)/base if base else np.nan})
price_volume_analysis(100,1_000_000,92,1_080_000)


## 7. Sales Force Budget & Incentive Design

In [ ]:
sales_force=pd.DataFrame({'Rep':[f'Rep_{i}' for i in range(1,21)],'Target':rng.uniform(8e6,18e6,20)})
sales_force['Achievement']=rng.uniform(.72,1.28,20); sales_force['Actual']=sales_force.Target*sales_force.Achievement
def incentive_multiplier(a):
    if a<.80:return 0
    if a<1:return .50
    if a<1.10:return 1.00
    if a<1.20:return 1.25
    return 1.50
sales_force['Incentive_Multiplier']=sales_force.Achievement.apply(incentive_multiplier)
sales_force['Bonus_Base']=sales_force.Target*.01
sales_force['Bonus']=sales_force.Bonus_Base*sales_force.Incentive_Multiplier
sales_force.head(10)


## 8. Marketing Budget & ROCI

In [ ]:
marketing=pd.DataFrame({'Initiative':['HCP Digital','Congress','KOL Program','Patient Awareness','Field Activation'],'Spend':[12e6,18e6,8e6,15e6,10e6],'Incremental_Net_Sales':[32e6,42e6,18e6,25e6,22e6],'Incremental_Contribution_Rate':[.45,.43,.46,.40,.44]})
marketing['Incremental_Contribution']=marketing.Incremental_Net_Sales*marketing.Incremental_Contribution_Rate
marketing['ROCI']=(marketing.Incremental_Contribution-marketing.Spend)/marketing.Spend
marketing.sort_values('ROCI',ascending=False)


## 9. Budget vs Actual Variance

In [ ]:
variance=finance.groupby(['Region','Channel'],as_index=False).agg(Budget=('Target_Gross_Sales','sum'),Actual=('Actual_Gross_Sales','sum'),Spend=('Commercial_Spend','sum'),Net_Sales=('Net_Sales','sum'),Contribution=('Contribution','sum'))
variance['Sales_Variance']=variance.Actual-variance.Budget
variance['Sales_Variance_%']=variance.Sales_Variance/variance.Budget
variance['Spend_%_of_Net_Sales']=variance.Spend/variance.Net_Sales
variance['ROCI']=(variance.Contribution-variance.Spend)/variance.Spend
variance.sort_values('Sales_Variance_%')


## 10. Reforecasting

In [ ]:
def reforecast_ytd(ytd_actual,months_elapsed,annual_target):
    run_rate=ytd_actual/max(months_elapsed,1); forecast=ytd_actual+run_rate*(12-months_elapsed)
    return pd.Series({'YTD Actual':ytd_actual,'Annual Target':annual_target,'Monthly Run Rate':run_rate,'FY Reforecast':forecast,'Gap vs Target':forecast-annual_target,'Forecast Achievement':forecast/annual_target if annual_target else np.nan})
reforecast_ytd(250_000_000,6,480_000_000)


## 11. Currency Impact

In [ ]:
def fx_impact(foreign_sales,old_fx,new_fx):
    old=foreign_sales*old_fx; new=foreign_sales*new_fx
    return pd.Series({'Foreign Currency Sales':foreign_sales,'Old FX':old_fx,'New FX':new_fx,'Old EGP Sales':old,'New EGP Sales':new,'FX Impact EGP':new-old,'FX Impact %':(new-old)/old if old else np.nan})
fx_impact(10_000_000,48,53)


## 12. Executive Commercial Decision Engine

In [ ]:
def commercial_decision(achievement,roci,commercial_spend):
    actions=[]
    if achievement<.90: actions.append('Review target realism, execution and regional/channel drivers.')
    elif achievement>=1.10: actions.append('Protect momentum; assess incremental investment using marginal ROCI.')
    if roci<.50: actions.append('Investigate commercial spend efficiency.')
    elif roci>1.50: actions.append('Consider scaling investment subject to marginal-return validation.')
    if commercial_spend>100_000_000: actions.append('Trigger commercial-spend deep dive and ROI validation.')
    return ' '.join(actions) if actions else 'Monitor performance.'
finance['Executive_Action']=finance.apply(lambda r:commercial_decision(r.Target_Achievement,r.ROCI,r.Commercial_Spend),axis=1)
finance[['Brand','Region','Channel','Target_Achievement','Commercial_Spend','ROCI','Executive_Action']].head(15)


## 13. FY2026 Scenario Planning

In [ ]:
scenario=pd.DataFrame({'Scenario':['Base','Upside','Downside'],'Target_Growth':[0,.10,-.08],'GTN_Rate':[.12,.115,.135],'COGS_Rate':[.38,.37,.40],'Commercial_Spend':[105.6e6,118e6,98e6]})
scenario['Gross_Target']=480e6*(1+scenario.Target_Growth); scenario['GTN']=scenario.Gross_Target*scenario.GTN_Rate; scenario['Net_Sales']=scenario.Gross_Target-scenario.GTN
scenario['COGS']=scenario.Net_Sales*scenario.COGS_Rate; scenario['Contribution']=scenario.Net_Sales-scenario.COGS; scenario['Contribution_After_Spend']=scenario.Contribution-scenario.Commercial_Spend; scenario['ROCI']=scenario.Contribution_After_Spend/scenario.Commercial_Spend
scenario


## 14. Export Outputs

In [ ]:
output_dir=os.path.join(DATA_DIR,'commercial_finance'); os.makedirs(output_dir,exist_ok=True)
finance.to_parquet(os.path.join(output_dir,'fact_commercial_finance.parquet'),index=False)
regional.to_csv(os.path.join(output_dir,'regional_budget_allocation.csv'),index=False)
channels_econ.to_csv(os.path.join(output_dir,'channel_economics.csv'),index=False)
sales_force.to_csv(os.path.join(output_dir,'sales_force_incentives.csv'),index=False)
marketing.to_csv(os.path.join(output_dir,'marketing_roci.csv'),index=False)
variance.to_csv(os.path.join(output_dir,'budget_vs_actual_variance.csv'),index=False)
scenario.to_csv(os.path.join(output_dir,'fy2026_budget_scenarios.csv'),index=False)
print('Saved:',output_dir)


## 15. Backend / Copilot Integration

Recommended module: `src/commercial_finance.py`

Expose: `build_budget`, `gtn_net_price`, `price_volume_analysis`, `target_cascade`, `channel_economics`, `calculate_incentives`, `calculate_marketing_roci`, `budget_vs_actual`, `reforecast_ytd`, `fx_impact`, `commercial_decision`.

This project should power the PharmaLens **Strategize → Commercial Finance** workspace and AI Copilot.